# 1. Initializations

## 1.1 General imports

In [ ]:
### global
from typing import cast
import logging
from smartcheck.logger_config import setup_logger
setup_logger(logging.INFO)

### data & text manipulation
import numpy as np
import pandas as pd
from collections import Counter
from PIL import Image
import cv2 
import re
import nltk
from nltk.tokenize import PunktSentenceTokenizer, word_tokenize
from nltk.tokenize.regexp import RegexpTokenizer
from nltk.corpus import stopwords
from nltk.stem.snowball import FrenchStemmer
from nltk.stem import WordNetLemmatizer
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

# machine learning
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

### graphical
import matplotlib.pyplot as plt
%matplotlib inline
from wordcloud import WordCloud
import seaborn as sns 

nltk.download('punkt_tab')
nltk.download('stopwords')
nltk.download('wordnet')

In [ ]:
import smartcheck.dataframe_common as dfc
import smartcheck.paths as pth

# 2. Loading and Data Enrichment

In [ ]:
df_movie_raw = dfc.load_dataset_from_config('movie_data', sep=',')

if df_movie_raw is not None and isinstance(df_movie_raw, pd.DataFrame):
    df_movie = df_movie_raw.copy()

In [ ]:
df_movie.info()

# 3. Text Mining

#### Regular expression parser

In [ ]:
txt = 'g.petoit93@gmail.com oliver.small459@orange.fr \n m.lameinère@yahoo.fr'

### Insérez votre code
r_mail = re.compile(r"""
   [\w\.\-è]+               #pseudonyme de l'adresse
   @                        #séparateur du domaine (obligatoire)
   [a-z\.]+                 #domaine de l'adresse
""", re.VERBOSE)
emails = r_mail.findall(txt)
print(emails)

#### Punkt Tokenizer (punkt_tab+stopwords)

In [ ]:
txt = "Souffrez qu'Amour cette nuit vous réveille. Par mes soupirs laissez-vous enflammer. "\
"Vous dormez trop, adorable merveille. Car c'est dormir que de ne point aimer."
punkt_tokenizer = PunktSentenceTokenizer()
punkt_tokenizer.tokenize(txt)
tkn_txt = word_tokenize(txt, language='french')
print(tkn_txt)

stop_words = set(stopwords.words('french'))
print(stop_words)
stop_words.add(".")
stop_words.update([".",","])
stop_words.discard(",")
stop_words.add(",")

# def stop_words_filtering(mot_vides, liste_mots):
#     liste_sans_mots_vides = []
#     for mot in liste_mots:
#         if mot not in mot_vides:
#             liste_sans_mots_vides.append(mot)
#     return liste_sans_mots_vides
def stop_words_filtering(mot_vides, liste_mots):
    return [mot for mot in liste_mots if mot not in mot_vides]
mots_filtres = stop_words_filtering(stop_words, tkn_txt)
print(mots_filtres)

#### RegExp Tokenizer

In [ ]:
regexp_tokenizer = RegexpTokenizer("[a-zé]{4,}")
tkn_txt = regexp_tokenizer.tokenize(txt.lower())
print(tkn_txt)

#### CountVectorizer

In [ ]:
cnt_vectorizer = CountVectorizer()
tokens = cnt_vectorizer.fit_transform(tkn_txt)
voc_txt = cnt_vectorizer.vocabulary_
print(voc_txt)
voc_txt = cast(np.array, cnt_vectorizer.transform([
    "laissez-vous enflammer",
    "Dormez vous cette nuit ?",
    "Dormez vous vous cette nuit ?"
])).toarray()
print(voc_txt)

In [ ]:
tfid_vectorizer = TfidfVectorizer()
tokens = tfid_vectorizer.fit_transform(tkn_txt)
voc_txt = tfid_vectorizer.vocabulary_
print(voc_txt)
voc_txt = cast(np.array, tfid_vectorizer.transform([
    "laissez-vous enflammer",
    "Dormez vous cette nuit ?",
    "Dormez vous vous cette nuit ?"
])).toarray()
print(voc_txt)

#### Racinisation

In [ ]:
french_stemmer = FrenchStemmer()
radical = french_stemmer.stem('sérieusement')
print(radical)

#### Lemmatisation

In [ ]:
wn_lemmatizer = WordNetLemmatizer()
print("lemmatisation verbe de meeting :", wn_lemmatizer.lemmatize('meeting', pos='v'))
print("lemmatisation nom de meeting :", wn_lemmatizer.lemmatize('meeting', pos='n'))

#### Nuage de mots

In [ ]:
def plot_word_cloud(text, masque, wc: WordCloud, figsize=(10,8)):
    # Appliquer le masque
    wc.mask = masque
    # Générer et afficher le nuage de mots
    plt.figure(figsize=figsize)
    wc.generate(text)
    plt.imshow(wc)
    plt.show()

In [ ]:
# extraction du texte
stop_words = set(stopwords.words('english'))
stop_words.update(["mission", "impossible", "harry", "potter", "Da", "Vinci", "Mountain", "Brokeback", "Code"])
print(stop_words)
df_pos = df_movie[df_movie.Sentiment==1]
df_neg = df_movie[df_movie.Sentiment==0]
pos_text = str([text.lower() for text in df_pos.Text])
neg_text = str([text.lower() for text in df_neg.Text])

# définition des sommet de la forme souhaité du masque (un losange ici)
sommets = np.array([
    [0, 200],
    [200, 0],
    [400, 200],
    [200, 400],
])
# creation du masque basé sur une trame de départ saturée (pixel blanc = 255) 
# sur laquelle on va appliquer la forme souhaité passante (pixel noir = 0)
full = np.ones(shape=(400,400))*255
mask = cv2.fillPoly(img=full, pts=[sommets], color=(0,))

# creation d'un wordcloud de sentiment positifs
pos_wc = WordCloud(background_color="white", max_words=100, stopwords=stop_words, max_font_size=50, random_state=1)
plot_word_cloud(pos_text, mask, pos_wc)

# creation d'un wordcloud de sentiment négatifs
neg_wc = WordCloud(background_color="black", max_words=100, stopwords=stop_words, max_font_size=50, random_state=1)
plot_word_cloud(neg_text, mask, neg_wc)

dico = Counter(neg_text.split())
mots = [m[0] for m in dico.most_common(15)]
freq = [m[1] for m in dico.most_common(15)]
plt.figure(figsize= (10,6))
sns.barplot(x=mots, y=freq)
plt.title('15 mots les plus fréquemment employés par les internautes laissant des mauvais commentaires')
plt.show()

# 4. Machine learning

#### Avec Counter Vectorizer

In [ ]:
# affichage clarifié des éléments un a un d'une colonne textuelle d'un dataframe (sans restriction d'affichage classique du display)
content = [i for i in df_movie['Text']]
print('\n'.join(content))

In [ ]:
X, y = df_movie.Text, df_movie.Sentiment
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state = 30)

In [ ]:
# Vectorisation avec counting
tfidf_vectorizer = CountVectorizer()
X_train = tfidf_vectorizer.fit_transform(X_train)
X_test = tfidf_vectorizer.transform(X_test)

In [ ]:
# Entrainement et prédiction du modèle
clf = GradientBoostingClassifier(n_estimators=100, learning_rate=1.0, max_depth=1, random_state=0)
clf.fit(X_train, y_train)
y_pred = clf.predict(X_test)

In [ ]:
# Rapport de classification
print(classification_report(y_test, y_pred))
# Calculer la matrice de confusion
cnf_matrix = confusion_matrix(y_test, y_pred, normalize='true')
# Tracer la heatmap de la matrice de confusion
plt.figure(figsize=(8, 6))
plt.title("Matrice de confusion")
sns.heatmap(cnf_matrix, cmap='Blues', annot=True, cbar=False, fmt=".2f")
plt.ylabel('Vrais labels')
plt.xlabel('Labels prédits')
plt.show()

#### Avec TF-IDF Vectorizer & Lemmatisation

In [ ]:
# utilitaires
def dfcol_list_to_string(df, col):
    df[col] = df[col].apply(lambda x: ' '.join(x))
    return df
def series_list_to_string(series):
    series = series.apply(lambda x: ' '.join(x))
    return series
def stop_words_filtering(mot_vides, liste_mots):
    return [mot for mot in liste_mots if mot not in mot_vides]
def lemmatisation(mots):
    wn_lemmatizer = WordNetLemmatizer()
    return list({wn_lemmatizer.lemmatize(mot) for mot in mots})

In [ ]:
# split features / target
X, y = df_movie.Text, df_movie.Sentiment
# regex manipulation
X = X.apply(lambda x: re.sub(r"\.+", '', x)) # effacement . / .. / ... / etc
X = X.apply(lambda x: re.sub(r"[0-9]+", '', x)) # effacement chiffres et nombres
# prérequis pour appliquer les stopwords/lemmatisation/etc... avoir splitté les phrases en mots
X = X.str.split() 
# stop words
stop_words = set(stopwords.words('english'))
extra_stop_words = [",", ".", "``", "@", "*", "(", ")", "...", "!", "?", "-", "_", ">", "<", ":", "/", "=", "--", "©", "~", ";", "\\", "\\\\"]
stop_words.update(extra_stop_words)
X = X.apply(lambda x: stop_words_filtering(stop_words, x))
# Lemmatisation
X = X.apply(lambda x: lemmatisation(x))
# concaténation (inverse du split)
X = series_list_to_string(X)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state = 30)

In [ ]:
print("\n".join(X.values))

In [ ]:
# Vectorisation avec counting
tfidf_vectorizer = TfidfVectorizer()
X_train = tfidf_vectorizer.fit_transform(X_train)
X_test = tfidf_vectorizer.transform(X_test)

In [ ]:
# Entrainement et prédiction du modèle
clf = GradientBoostingClassifier(n_estimators=100, learning_rate=1.0, max_depth=1, random_state=0)
clf.fit(X_train, y_train)
y_pred = clf.predict(X_test)

In [ ]:
# Rapport de classification
print(classification_report(y_test, y_pred))
# Calculer la matrice de confusion
cnf_matrix = confusion_matrix(y_test, y_pred, normalize='true')
# Tracer la heatmap de la matrice de confusion
plt.figure(figsize=(8, 6))
plt.title("Matrice de confusion")
sns.heatmap(cnf_matrix, cmap='Blues', annot=True, cbar=False, fmt=".2f")
plt.ylabel('Vrais labels')
plt.xlabel('Labels prédits')
plt.show()